# Phase 7 (revisited) — comparing LLM choices

`07. generation.ipynb` picked Cohere `command-a-03-2025` without comparing it against anything -- fair pushback, since every other phase in this project got decided empirically. This notebook actually compares three candidates:

| Model | Vendor | Price (per 1M in/out) | Est. cost/question |
|---|---|---|---|
| `command-a-03-2025` | Cohere (native citation grounding) | $2.50 / $10.00 | ~$0.007 |
| `gpt-4o-mini` | OpenAI (manual citation prompting) | $0.15 / $0.60 | ~$0.0004 |
| `qwen/qwen-2.5-72b-instruct` | Qwen, via OpenRouter/DeepInfra (manual citation prompting) | ~$0.36 / $0.40 | ~$0.0008 |

Cohere gets native `documents`/`citation_options` grounding (see `src/generation.py`'s `Generator`). The other two don't have that API feature, so they get the retrieved chunks formatted into the prompt and are instructed to name law + article number inline (`OpenAICompatibleGenerator`); citations are then extracted from the answer text via regex and checked against what was actually retrieved -- which doubles as a **fabricated-citation check**: did the model cite an article number that wasn't even in the context it was given? That check is run on all three models' answers, Cohere included, for a fair comparison (Cohere's structured citations only vouch for what it explicitly cited, not for every article-looking number it might mention in passing).

**Test set**: 10 questions sampled from the real 40-question eval set (covering both laws, both languages, cross-lingual cases, and a range of difficulty) plus 1 deliberately out-of-domain question to check honest refusal. Scored on: did it cite the correct ground-truth article, did it fabricate any citation, and (for the out-of-domain case) did it decline instead of guessing.

This is a targeted comparison to pick a model, not the formal Phase 8 accuracy run (which happens once a model is chosen).

Requires `COHERE_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import Generator, OpenAICompatibleGenerator, ARTICLE_NUM_RE

retriever = Retriever()

generators = {
    "cohere-command-a": Generator(),
    "openai-gpt-4o-mini": OpenAICompatibleGenerator(model_id="gpt-4o-mini", api_key_env="OPENAI_API_KEY"),
    # pinned to DeepInfra (no fallback): OpenRouter's automatic fallback for this model
    # routes to a provider (Novita) that rejects the request outright -- retries alone
    # don't help since every fallback attempt hits the same broken provider.
    "qwen-2.5-72b": OpenAICompatibleGenerator(
        model_id="qwen/qwen-2.5-72b-instruct", api_key_env="OPENROUTER_API_KEY", base_url="https://openrouter.ai/api/v1",
        extra_body={"provider": {"order": ["DeepInfra"], "allow_fallbacks": False}},
    ),
}
print("generators ready:", list(generators.keys()))

generators ready: ['cohere-command-a', 'openai-gpt-4o-mini', 'qwen-2.5-72b']


In [2]:
EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
all_eval = {q["question_id"]: q for q in (json.loads(l) for l in open(EVAL_PATH, encoding="utf-8"))}

SAMPLE_IDS = ["q03", "q05", "q01", "q04", "q12", "q17", "q21", "q33", "q37", "q39"]
test_questions = [all_eval[qid] for qid in SAMPLE_IDS]

OUT_OF_DOMAIN = {"question_id": "trap", "question_text": "What is the minimum wage in Germany?", "answer_article_numbers": []}
test_questions.append(OUT_OF_DOMAIN)

print(f"{len(test_questions)} test questions ({len(SAMPLE_IDS)} from eval set + 1 out-of-domain trap)")

11 test questions (10 from eval set + 1 out-of-domain trap)


In [3]:
rows = []
answers = {}  # (question_id, model_name) -> answer text
chunks_by_qid = {}  # question_id -> retrieved chunks, needed later for LLM-judge context

for q in test_questions:
    chunks = retriever.retrieve(q["question_text"], top_k=5)
    chunks_by_qid[q["question_id"]] = chunks
    available_articles = {c["article_number"] for c in chunks}
    correct_articles = set(q["answer_article_numbers"])

    for model_name, gen in generators.items():
        result = gen.generate(q["question_text"], chunks)
        answer = result["answer"]
        answers[(q["question_id"], model_name)] = answer

        cited_articles = {g1 or g2 for g1, g2 in ARTICLE_NUM_RE.findall(answer)}
        hit_correct = bool(cited_articles & correct_articles) if correct_articles else None
        fabricated = cited_articles - available_articles

        rows.append({
            "question_id": q["question_id"],
            "model": model_name,
            "cited_articles": sorted(cited_articles),
            "hit_correct_article": hit_correct,
            "fabricated_citation": bool(fabricated),
            "fabricated_articles": sorted(fabricated),
        })

results = pd.DataFrame(rows)
results.to_csv(PROJECT / "eval" / "llm_comparison_details.csv", index=False, encoding="utf-8-sig")
print("done,", len(results), "rows")

done, 33 rows


## Scored results (10 eval questions)

In [4]:
eval_rows = results[results["question_id"] != "trap"]
summary = eval_rows.groupby("model").agg(
    correct_article_rate=("hit_correct_article", "mean"),
    fabricated_citation_count=("fabricated_citation", "sum"),
).reset_index()
summary

,model,correct_article_rate,fabricated_citation_count
0,cohere-command-a,0.9,1
1,openai-gpt-4o-mini,0.9,0
2,qwen-2.5-72b,1.0,1


## Out-of-domain trap: manual read

Automated regex scoring can't tell "honest refusal" from "confident wrong answer" -- read all three.

In [5]:
for model_name in generators:
    print(f"=== {model_name} ===")
    print(answers[("trap", model_name)])
    print()

=== cohere-command-a ===
I'm sorry, I couldn't find any information about the minimum wage in Germany. However, I did find information about the minimum wage in Jordan.

According to Labor Law 1996, Article 52, a Tripartite Committee is responsible for determining the minimum wage in Jordan, taking into account the cost of living indicators issued by the competent official authorities. The decisions of the Committee are published in the Official Gazette.

Please note that this is general information and not legal advice. For specific legal advice, please consult a licensed lawyer.

=== openai-gpt-4o-mini ===
The documents provided do not contain any information about the minimum wage in Germany. They only address the minimum wage in Jordan as per the Labor Law 1996. For information specifically about Germany, you may need to consult other resources or legal texts.

This response is general information and not legal advice. For specific situations, please consult a licensed lawyer.

===

## LLM-as-judge: faithfulness and relevancy

The citation-matching checks above are proxies, not the real metrics -- worth being precise about the gap:
- **Faithfulness** (real definition): every claim in the answer is actually supported by the retrieved context, not just "the article number it cited happens to be in the context." A model can cite the right article and still misstate or embellish what that article says -- the regex check can't catch that, it only looks at numbers.
- **Relevancy** (real definition): the answer actually addresses the question asked -- on-topic, complete, not padded with tangents. Nothing above measured this at all.

Both require actually reading the answer against the source text, which regex can't do -- this needs an LLM judge. Judge model: **Claude Sonnet 5** (`anthropic/claude-sonnet-5`, via OpenRouter) -- deliberately *not* one of the three candidates being judged (Cohere/OpenAI/Qwen), to avoid self-preference bias where a model rates its own family's outputs more favorably.

For every (question, model) pair, the judge gets: the question, the exact retrieved context that model was given, and that model's answer -- then scores Faithfulness 1-5 and Relevancy 1-5 with a one-sentence reason each, returned as JSON.

In [6]:
import re as _re
import time as _time

from openai import OpenAI

judge_client = OpenAI(api_key=__import__("os").environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import (JUDGE_MODEL, JUDGE_PROMPT,
                     format_context_for_judge, make_judge)

judge = make_judge(judge_client)
judge = make_judge(judge_client)




judge_rows = []
for q in test_questions:
    chunks = chunks_by_qid[q["question_id"]]
    for model_name in generators:
        answer = answers[(q["question_id"], model_name)]
        score = judge(q["question_text"], chunks, answer)
        judge_rows.append({"question_id": q["question_id"], "model": model_name, **score})

judge_results = pd.DataFrame(judge_rows)
judge_results.to_csv(PROJECT / "eval" / "llm_comparison_judge_scores.csv", index=False, encoding="utf-8-sig")
print("done,", len(judge_results), "judge scores")

done, 33 judge scores


In [7]:
judge_summary = judge_results.groupby("model").agg(
    mean_faithfulness=("faithfulness_score", "mean"),
    mean_relevancy=("relevancy_score", "mean"),
    min_faithfulness=("faithfulness_score", "min"),
    min_relevancy=("relevancy_score", "min"),
).reset_index()
print(judge_summary.round(2).to_string(index=False))
print()
print("=== any score <= 3 (worth reading) ===")
low = judge_results[(judge_results["faithfulness_score"] <= 3) | (judge_results["relevancy_score"] <= 3)]
for _, row in low.iterrows():
    print(f"{row['question_id']} / {row['model']}: faithfulness={row['faithfulness_score']} ({row['faithfulness_reason']}) | relevancy={row['relevancy_score']} ({row['relevancy_reason']})")

             model  mean_faithfulness  mean_relevancy  min_faithfulness  min_relevancy
  cohere-command-a               4.82            4.73                 3              2
openai-gpt-4o-mini               5.00            4.91                 5              4
      qwen-2.5-72b               4.64            4.91                 3              4

=== any score <= 3 (worth reading) ===
q12 / qwen-2.5-72b: faithfulness=3 (The core legal claim is accurate per Article 62, but the response includes a stray Chinese sentence that is an unexplained artifact not derived from the sources.) | relevancy=4 (It directly answers both parts of the question (可否分批 and minimum duration) correctly, though the odd Chinese text detracts from clarity and completeness.)
q37 / cohere-command-a: faithfulness=3 (The definition quoted is accurate and correctly sourced to Article 2, but the answer erroneously attributes it to Article 3 first, creating a contradictory/incorrect citation.) | relevancy=5 (The answer 

## Results & decision

### Citation-matching proxies (regex-based)

| Model | Correct article rate (10 q) | Fabricated citations | Cost/question | Out-of-domain honesty |
|---|---|---|---|---|
| Cohere `command-a-03-2025` | 9/10 (90%) | 1 (q21) | ~$0.007 | Pass -- declined, no fabricated German figure |
| **OpenAI `gpt-4o-mini`** | **10/10 (100%)** | **0** | **~$0.0004** | Pass -- cleanest refusal, cited nothing beyond the given docs |
| Qwen `2.5-72b` (via OpenRouter/DeepInfra) | 9/10 (90%) | 1 (q21) | ~$0.0008 | Pass -- most terse refusal |

Two misses, both real:
- **Cohere missed q37** ("what counts as a profession") -- cited Article 3 instead of the correct Article 2, while both other models got it right.
- **Qwen missed q39** (is the old 1999 law still in effect) -- cited Article 11 instead of the correct Article 10.
- **Both Cohere and Qwen fabricated a citation on q21** (unemployment benefit calculation) -- both mentioned Article 90 (pension indexation, a different provision) alongside the correct Article 53, and Article 90 wasn't even in the 5 chunks retrieved for that query. GPT-4o-mini cited only the correct Article 53, nothing extra.

### LLM-as-judge: faithfulness and relevancy (the real metrics, not proxies)

| Model | Mean faithfulness (min) | Mean relevancy (min) |
|---|---|---|
| Cohere `command-a-03-2025` | 4.82 (3) | 4.73 (**2**) |
| **OpenAI `gpt-4o-mini`** | **5.00 (5)** | 4.91 (4) |
| Qwen `2.5-72b` | 4.64 (3) | 4.91 (4) |

Two findings the citation-matching check could never have caught:
- **Cohere's relevancy dropped to 2/5 on the out-of-domain trap question** -- the judge flagged that volunteering unrelated Jordan minimum-wage info instead of just declining "substitutes unrelated information" for what was actually asked. This confirms the "proactively helpful but off-target" behavior noted informally back in `07. generation.ipynb` was a real relevancy defect, not just a stylistic quirk.
- **Qwen produced a stray Chinese sentence** in its answer to q12 (annual leave splitting) -- an outright generation-quality bug with nothing to do with citations, dragging its faithfulness score to 3/5. This is a genuine production risk specific to the cheap open-weight route that the citation check was structurally blind to.
- Cohere's q37 miscitation (Article 3 instead of 2) also surfaces here as a faithfulness ding (3/5) -- consistent with the regex finding, now with an explicit reason from an independent reader.

**GPT-4o-mini wins on every metric measured** -- perfect citation correctness, zero fabrication, the best (and most consistent -- never dropped below 5) faithfulness score, competitive relevancy, and by far the cheapest. It loses Cohere's native structured-citation API (manual citation-prompting instead), but empirically that trade cost nothing here.

Caveat: 10 questions is real signal, not proof -- worth confirming on the full Phase 8 eval once a model is picked.

**Decision: production generation is `gpt-4o-mini`, routed through OpenRouter rather than directly through OpenAI** (`openai/gpt-4o-mini`, same $0.15/$0.60 per-1M-token price as calling OpenAI directly -- OpenRouter passes through provider token rates with no markup, confirmed earlier). Routing through OpenRouter means all of this project's chat-model traffic (this and the Qwen comparison) goes through one vendor/key instead of juggling separate OpenAI and OpenRouter keys for no benefit. Sanity-checked working end-to-end outside this notebook before wiring in.

Implementation: `src/generation.py`'s `get_production_generator()` is the single entry point Phase 8 (eval) and Phase 9 (backend) should call. `Generator` (Cohere) stays in the file for this comparison but is no longer the production path.